# Reference uniform-FMM upward pass

This notebook visualises how one selected particle contributes to the upward pass:

```text
selected particle
    ↓ P2M
containing leaf multipole
    ↓ M2M
ancestor multipoles
    ↓ M2M
root multipole
```

The complete uniform tree remains visible for context. Select a particle to highlight its dipole moment, containing leaf, and every ancestor node through which its contribution passes. Other particles and tree nodes are faded.

A Cartesian multipole is a collection of coefficients rather than a single vector. The highlighted arrows therefore show the **upward transfer route** of the selected particle's contribution, not an orientation of the complete multipole expansion. No value or error metric is used because this notebook isolates only the upward pass.


## Random particles and unit dipole moments

Positions are drawn uniformly inside the cubic root region. Dipole moments are drawn independently and normalised to unit length. Change the parameters below and rerun the notebook to generate another deterministic configuration.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import IntSlider, interact

import cdfmm

try:
    from example_utils import (
        draw_box_3d,
        finish_3d_axes,
        random_unit_vectors,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        draw_box_3d,
        finish_3d_axes,
        random_unit_vectors,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

n_particles = 50
random_seed = 42
root_half_width = 1.0
max_level = 3
expansion_order = 5

rng = np.random.default_rng(random_seed)
positions = rng.uniform(
    low=-root_half_width,
    high=root_half_width,
    size=(n_particles, 3),
)
moments = random_unit_vectors(rng, n_particles)

options = cdfmm.UniformFmmOptions()
options.expansion_order = expansion_order
options.tree.max_level = max_level
options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
options.tree.root_half_width = root_half_width
options.fixed_target_source_indices = []

fmm = cdfmm.UniformFmm(positions, options)
fmm.upward_pass(moments)

print(f"Particles: {n_particles}")
print(f"Moment lengths: {np.linalg.norm(moments, axis=1).min():.6f} to "
      f"{np.linalg.norm(moments, axis=1).max():.6f}")
print(f"Tree levels: {fmm.tree.n_levels}; total nodes: {len(fmm.tree.nodes)}")


## Interactive particle path

Use the slider to choose a particle in its original input order. Its purple arrow is the unit dipole moment. The gold P2M arrow leads to the containing leaf centre; red M2M arrows then follow the contribution from that leaf to the root. The strongly outlined boxes are exactly the nodes touched along this path.


In [ ]:
def upward_path_for_particle(particle_index):
    tree = fmm.tree
    nodes = tree.nodes

    # Tree leaf lookup uses Morton-sorted indices, while the selector uses
    # the original particle order supplied to UniformFmm.
    sorted_index = tree.source_inverse_permutation[particle_index]
    node_index = tree.leaf_index_for_source(sorted_index)

    path_indices = []
    while node_index >= 0:
        path_indices.append(node_index)
        node_index = nodes[node_index].parent

    return sorted_index, path_indices


def plot_particle_upward_path(particle_index=0):
    tree = fmm.tree
    nodes = tree.nodes
    sorted_index, path_indices = upward_path_for_particle(particle_index)
    path_set = set(path_indices)
    selected_position = positions[particle_index]
    selected_moment = moments[particle_index]
    leaf = nodes[path_indices[0]]

    figure = plt.figure(figsize=(10, 8))
    axes = figure.add_subplot(projection="3d")

    # Retain the complete octree as quiet geometric context.
    background_label_added = False
    for node in nodes:
        if node.index in path_set:
            continue
        draw_box_3d(
            axes,
            vec3_to_array(node.centre),
            node.half_width,
            colour="0.65",
            linewidth=0.35,
            alpha=0.10,
            label="other tree nodes" if not background_label_added else None,
        )
        background_label_added = True

    # Highlight the selected leaf and each ancestor up to the root.
    path_colours = plt.cm.autumn(np.linspace(0.15, 0.85, tree.n_levels))
    for node_index in reversed(path_indices):
        node = nodes[node_index]
        draw_box_3d(
            axes,
            vec3_to_array(node.centre),
            node.half_width,
            colour=path_colours[node.level],
            linewidth=2.0 if node.is_leaf else 1.5,
            alpha=0.95,
            label=f"level {node.level} path node",
        )

    other_particles = np.arange(n_particles) != particle_index
    axes.scatter(
        *positions[other_particles].T,
        s=18,
        color="tab:blue",
        alpha=0.18,
        label="other particles",
    )
    axes.quiver(
        *positions[other_particles].T,
        *moments[other_particles].T,
        length=0.12 * root_half_width,
        normalize=True,
        color="tab:blue",
        alpha=0.10,
        arrow_length_ratio=0.25,
    )

    axes.scatter(
        *selected_position,
        s=70,
        color="tab:purple",
        edgecolor="white",
        linewidth=0.8,
        label=f"selected particle {particle_index}",
        zorder=10,
    )
    axes.quiver(
        *selected_position,
        *selected_moment,
        length=0.28 * root_half_width,
        normalize=True,
        color="tab:purple",
        linewidth=2.4,
        arrow_length_ratio=0.22,
        label="selected unit moment",
    )

    # P2M forms the selected particle's leaf-centred contribution.
    leaf_centre = vec3_to_array(leaf.centre)
    p2m_displacement = leaf_centre - selected_position
    axes.quiver(
        *selected_position,
        *p2m_displacement,
        color="goldenrod",
        linewidth=2.8,
        arrow_length_ratio=0.16,
        label="P2M route",
    )

    # M2M translates the contribution from each child centre to its parent.
    for path_position, child_index in enumerate(path_indices[:-1]):
        child = nodes[child_index]
        parent = nodes[path_indices[path_position + 1]]
        child_centre = vec3_to_array(child.centre)
        parent_centre = vec3_to_array(parent.centre)
        axes.quiver(
            *child_centre,
            *(parent_centre - child_centre),
            color="tab:red",
            linewidth=3.0,
            arrow_length_ratio=0.14,
            label="M2M route" if path_position == 0 else None,
        )

    path_centres = np.array(
        [vec3_to_array(nodes[node_index].centre) for node_index in path_indices]
    )
    axes.scatter(
        *path_centres.T,
        s=45,
        color="tab:red",
        edgecolor="white",
        linewidth=0.7,
        zorder=9,
    )

    finish_3d_axes(
        axes,
        f"Upward path for particle {particle_index}: leaf to root",
    )
    axes.legend(loc="upper left", fontsize=8)
    figure.tight_layout()
    plt.show()

    print(f"Original particle index: {particle_index}")
    print(f"Morton-sorted index:     {sorted_index}")
    print(f"Unit moment:             {selected_moment}")
    print(f"Leaf Morton index:       {leaf.morton_index}")
    print(f"Touched node indices:    {path_indices}")


interact(
    plot_particle_upward_path,
    particle_index=IntSlider(
        min=0,
        max=n_particles - 1,
        step=1,
        value=0,
        description="Particle",
        continuous_update=False,
    ),
)


## What to observe

Every particle enters exactly one leaf through P2M and then contributes to one node at each coarser level through M2M. Particles in the same leaf share the complete highlighted path. Particles in different leaves may merge at a common ancestor, after which their remaining routes to the root are identical.
